# autoresearchABM — Experiment Analysis

Analysis of autonomous hyperparameter search results from `results.tsv`.

Primary metric: `val_pi` (peak infection prevalence) — **lower is better (flatter curve)**.
Secondary metric: `val_ar` (final attack rate) — also lower is better.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["val_pi"]    = pd.to_numeric(df["val_pi"],    errors="coerce")
df["seeds_run"] = pd.to_numeric(df["seeds_run"], errors="coerce")
df["status"]    = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep    = counts.get("KEEP",    0)
n_discard = counts.get("DISCARD", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  val_pi={row['val_pi']:.6f}  seeds={int(row['seeds_run'])}  {row['description']}")

## val_pi Over Time

Running minimum shows the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline_pi = valid.loc[0, "val_pi"]
best_pi = valid[valid["status"] == "KEEP"]["val_pi"].min()

below = valid[valid["val_pi"] <= baseline_pi + 0.02]

disc = below[below["status"] == "DISCARD"]
ax.scatter(disc.index, disc["val_pi"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

kept_v = below[below["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["val_pi"],
           c="#e74c3c", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

kept_mask = valid["status"] == "KEEP"
kept_idx  = valid.index[kept_mask]
kept_pi   = valid.loc[kept_mask, "val_pi"]
running_min = kept_pi.cummin()
ax.step(kept_idx, running_min, where="post", color="#c0392b",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

for idx, pi in zip(kept_idx, kept_pi):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, pi),
                textcoords="offset points", xytext=(6, 6),
                fontsize=8, color="#7b241c", alpha=0.9,
                rotation=30, ha="left", va="bottom")

n_total = len(df)
n_kept  = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Peak Infection Prevalence — val_pi (lower = flatter curve)", fontsize=12)
ax.set_title(f"autoresearchABM Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.2)

margin = (baseline_pi - best_pi) * 0.15
ax.set_ylim(best_pi - margin, baseline_pi + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline_pi = df.iloc[0]["val_pi"]
best_pi     = kept["val_pi"].min()
best_row    = kept.loc[kept["val_pi"].idxmin()]

print(f"Baseline val_pi:   {baseline_pi:.6f}")
print(f"Best val_pi:       {best_pi:.6f}")
print(f"Total improvement: {baseline_pi - best_pi:.6f}  ({(baseline_pi - best_pi) / baseline_pi * 100:.2f}%)")
print(f"Best experiment:   {best_row['description']}")
print()

print("Cumulative improvements:")
for _, row in kept.reset_index().iterrows():
    print(f"  Experiment #{int(row['index']):3d}: val_pi={row['val_pi']:.6f}  {row['description']}")

## Top Hits (by marginal improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_pi"] = kept["val_pi"].shift(1)
kept["delta"]   = kept["prev_pi"] - kept["val_pi"]

hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>9}  {'val_pi':>10}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_pi']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")